In [18]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv, find_dotenv


In [19]:
from langchain_openai import OpenAI
from langchain.callbacks import get_openai_callback


In [20]:
OPENAI_API_KEY=os.getenv('OPENAI_API_KEY')
HUGGINGFACEHUB_API_TOKEN=os.getenv('HUGGINGFACEHUB_API_TOKEN')
PINECONE_API_KEY=os.getenv('PINECONE_API_KEY')

In [21]:
llm = OpenAI(openai_api_key=OPENAI_API_KEY, temperature=0)

In [22]:
#token usage overview

def count_tokens(agent, query):
    with get_openai_callback() as cb:
        result = agent.invoke(query)
        print(f'Spent a total of {cb.total_tokens} tokens')
    return result


# Load and prepare dataframe

In [23]:
df = pd.read_csv('datasets/imdb_clean.csv')

df.head()

,Unnamed: 0,title,director,release_year,runtime,genre,rating,metascore,gross(M)
0,0,The Shawshank Redemption,Frank Darabont,1994,142,Drama,9.3,82,28.34
1,1,The Godfather,Francis Ford Coppola,1972,175,Crime,9.2,100,134.97
2,1,The Godfather,Francis Ford Coppola,1972,175,Drama,9.2,100,134.97
3,2,The Dark Knight,Christopher Nolan,2008,152,Action,9.0,84,534.86
4,2,The Dark Knight,Christopher Nolan,2008,152,Crime,9.0,84,534.86


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2532 entries, 0 to 2531
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0    2532 non-null   int64  
 1   title         2532 non-null   str    
 2   director      2532 non-null   str    
 3   release_year  2532 non-null   int64  
 4   runtime       2532 non-null   int64  
 5   genre         2532 non-null   str    
 6   rating        2532 non-null   float64
 7   metascore     2532 non-null   int64  
 8   gross(M)      2532 non-null   float64
dtypes: float64(2), int64(4), str(3)
memory usage: 178.2 KB


In [25]:
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"])
df.head()

,Unnamed: 0,title,director,release_year,runtime,genre,rating,metascore,gross(M)
0,0,The Shawshank Redemption,Frank Darabont,1994,142,Drama,9.3,82,28.34
1,1,The Godfather,Francis Ford Coppola,1972,175,Crime,9.2,100,134.97
2,1,The Godfather,Francis Ford Coppola,1972,175,Drama,9.2,100,134.97
3,2,The Dark Knight,Christopher Nolan,2008,152,Action,9.0,84,534.86
4,2,The Dark Knight,Christopher Nolan,2008,152,Crime,9.0,84,534.86


## Create SQL-Database from scratch

In [26]:
from sqlalchemy import create_engine
from langchain_community.utilities import SQLDatabase



In [27]:
engine = create_engine("sqlite:///:memory:")

In [28]:
#map dataframe in database
df.to_sql("imdb_movies", con=engine, index=False)

2532

In [29]:
db = SQLDatabase(engine)

In [30]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [31]:
agent_executor = create_sql_agent(
    llm=llm,
    toolkit=SQLDatabaseToolkit(db=db, llm=llm),
    verbose=True,
    #agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    max_iterations=5
)

In [32]:
response = agent_executor.invoke({"input": "Wie viele Filme gibt es in der Datenbank?"})
print(response["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Action: sql_db_list_tables
Action Input: imdb_moviesI should query the number of rows in the imdb_movies table to determine the total number of movies in the database.
Action: sql_db_query
Action Input: SELECT COUNT(*) FROM imdb_movies[(2532,)]I now know the final answer
Final Answer: There are 2532 movies in the database.

> Finished chain.
There are 2532 movies in the database.
